<a href="https://colab.research.google.com/github/aayushsinghiitgn/Pandas_Practice/blob/main/Pandas_group_by_assignments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pathlib import Path
import pandas as pd
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/GroupPPT")
AS_OF = pd.Timestamp("2026-10-09 23:59:59")
members = pd.read_csv(DATA_DIR / "members.csv", parse_dates=["join_date"])
renewals = pd.read_csv(DATA_DIR / "renewal_opportunities.csv", parse_dates=["expiry_date", "renewal_deadline"])
events = pd.read_csv(DATA_DIR / "member_events.csv", parse_dates=["event_time"])

# Date columns supplied for the exercises.
members["join_year"] = members["join_date"].dt.year
renewals["expiry_year"] = renewals["expiry_date"].dt.year
renewals["expiry_month"] = renewals["expiry_date"].dt.month
print(members.shape, renewals.shape, events.shape)

Mounted at /content/drive
(8580, 9) (11580, 9) (152057, 10)


In [3]:
members.head()

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023


In [4]:
renewals.head()

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9


In [5]:
events.head()

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13


### 1

Using `members`, group by `acquisition_channel`. Return:

- `member_count`: number of members.
- `average_annual_fee`: mean annual fee.
- `earliest_join`: earliest joining date.
- `latest_join`: latest joining date.

In [10]:
members.groupby("acquisition_channel").agg(
    member_count=("member_id", "count"),
    average_annual_fee=("annual_fee", "mean"),
    earliest_join=("join_date", "min"),
    latest_join=("join_date", "max")
)

,member_count,average_annual_fee,earliest_join,latest_join
acquisition_channel,,,,
APP,4328,1805.839187,2023-09-01,2026-10-01
STORE,1655,1799.302115,2023-09-01,2026-10-01
WEB,2597,1794.725838,2023-09-01,2026-10-01


Using `.loc`, select members whose `status_2026_10_09` is `"PAID"`. Group them by `city` and return:

- `paid_members`: number of members.
- `average_annual_fee`: mean annual fee.
- `channels_present`: number of distinct acquisition channels.

In [14]:
members.loc[members['status_2026_10_09'] == 'PAID'].groupby('city').agg(paid_members = ('member_id', 'count'),
                                                                        average_annual_fee = ('annual_fee','mean'),
                                                                        channels_present = ('acquisition_channel', 'nunique')
)

,paid_members,average_annual_fee,channels_present
city,,,
Ahmedabad,725,1799.689655,3
Bengaluru,771,1821.957198,3
Delhi,754,1786.798408,3
Gurugram,755,1782.443709,3
Hyderabad,789,1800.647655,3
Jaipur,778,1819.051414,3
Mumbai,754,1772.209549,3
Pune,698,1794.128940,3


Select members who were `"PAID"` on 9 October 2025 but `"EXPIRED"` on 9 October 2026. Group by `acquisition_channel` and return:

- `expired_members`: number of members.
- `average_annual_fee`: mean annual fee.

Use `.loc` on the aggregated result to display only channels with at least 300 such members.

In [52]:
mem_agg = members.loc[
    (members["status_2025_10_09"] == "PAID") &
    (members["status_2026_10_09"] == "EXPIRED")
]


memr = mem_agg.groupby('acquisition_channel').agg(expired_members = ('member_id','count'), average_annual_fee = ('annual_fee', 'mean'))
mem = memr.loc[memr['expired_members'] >= 300]
mem

,expired_members,average_annual_fee
acquisition_channel,,
APP,619,1794.638126
WEB,391,1826.365729


Select members who joined in 2025 or 2026 and are `"PAID"` at the current snapshot. Group by both `join_year` and `acquisition_channel`. Return:

- `paid_members`: number of members.
- `cities_present`: number of distinct cities.
- `average_annual_fee`: mean annual fee.

In [45]:
members.loc[
    members['join_year'].isin([2025, 2026])  # Use [2025, 2026] if join_year is numeric
    & (
        (members['status_2025_10_09'] == 'PAID')
        | (members['status_2026_10_09'] == 'PAID')
    )
].groupby(['join_year', 'acquisition_channel']).agg(
    paid_members=('member_id', 'count'),
    cities_present=('city', 'nunique'),
    average_annual_fee=('annual_fee', 'mean')
)

paid_members  cities_present  \
join_year acquisition_channel                                 
2025      APP                          1143               8   
          STORE                         432               8   
          WEB                           689               8   
2026      APP                           924               8   
          STORE                         351               8   
          WEB                           541               8   

                               average_annual_fee  
join_year acquisition_channel                      
2025      APP                         1792.088364  
          STORE                       1843.907407  
          WEB                         1789.275762  
2026      APP                         1807.441558  
          STORE                       1795.296296  
          WEB                         1794.748614


Select currently paid members acquired through `"APP"` or `"WEB"`. Group by `city` and return:

- `paid_members`: number of members.
- `plans_present`: number of distinct plans.
- `average_annual_fee`: mean annual fee.

Using `.loc` on the summary, display only cities with at least 500 paid members and an average annual fee above 1,750. Show only `paid_members` and `average_annual_fee` in the final output.


In [59]:
mem_agg = members.loc[(members['status_2026_10_09'] == 'PAID') & ((members['acquisition_channel'] == 'APP') | (members['acquisition_channel'] == 'WEB'))].groupby('city').agg(paid_members = ('member_id','count'), plans_present = ('plan_code', 'nunique'), average_annual_fee = ('annual_fee', 'mean'))

mem = mem_agg.loc[(mem_agg['paid_members'] >= 500) & (mem_agg['average_annual_fee'] > 1750), ['paid_members' , 'average_annual_fee'] ]
mem

,paid_members,average_annual_fee
city,,
Ahmedabad,577,1805.759099
Bengaluru,625,1823.800000
Delhi,613,1784.481240
Gurugram,601,1783.525790
Hyderabad,629,1804.246423
Jaipur,646,1825.625387
Mumbai,604,1765.556291
Pune,572,1790.958042


In [60]:
renewals.head()

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9


### 6

Using `renewals`, group by `renewal_number` and return:

- `members_due`: number of renewal records.
- `average_fee`: mean renewal fee.
- `average_discount`: mean offered discount, including zero discounts.
- `maximum_discount`: largest offered discount.

These results describe offers and members due for renewal, not successful renewals.

In [61]:
renewals.groupby('renewal_number').agg(members_due = ('renewal_id', 'count'), average_fee = ('renewal_fee', 'mean'), average_discount = ('offered_discount', 'mean'), maximum_discount = ('offered_discount', 'max'))

,members_due,average_fee,average_discount,maximum_discount
renewal_number,,,,
1,6380,1801.194357,14.490596,250
2,3400,1800.764706,19.205882,250
3,1800,1809.000000,21.083333,250


### 7

Select renewal records with expiry month September and expiry year 2025 or 2026. Group by `expiry_year` and return:

- `members_due`: number of renewal records.
- `distinct_members`: number of distinct member IDs.
- `first_expiry`: earliest expiry date.
- `last_expiry`: latest expiry date.

Check whether `members_due` and `distinct_members` agree in each year. What would a disagreement mean?

In [64]:
renewals[(renewals['expiry_month'] == 9) &
         ((renewals['expiry_year'] == 2025 ) | (renewals['expiry_year'] == 2026))].groupby('expiry_year').agg(members_due = ('renewal_id','count'),
                                                                                                              distinct_members = ('member_id', 'nunique'),
                                                                                                              first_expiry = ('expiry_date', 'min'),
                                                                                                              last_expiry = ('expiry_date', 'max'))

,members_due,distinct_members,first_expiry,last_expiry
expiry_year,,,,
2025,4000,4000,2025-09-01,2025-09-30
2026,4480,4480,2026-09-01,2026-09-30


### 8

Select September renewal records from 2025 and 2026. Group by both `expiry_year` and `renewal_number`. Return:

- `members_due`: number of renewal records.
- `average_fee`: mean renewal fee.
- `total_offered_discount`: sum of offered discounts.

The discount total is the sum of amounts offered across records; it is not proof that all those discounts were redeemed.

In [68]:
renewals.loc[(renewals['expiry_month'] == 9) & ((renewals['expiry_year'] == 2025) | (renewals['expiry_year'] == 2026))].groupby(['expiry_year', 'renewal_number']).agg(members_due = ('renewal_id', 'count'),
                                                                                                                                 average_fee = ('renewal_fee', 'mean'),
                                                                                                                                 total_offered_discount = ('offered_discount', 'sum'))

members_due  average_fee  total_offered_discount
expiry_year renewal_number                                                  
2025        1                      2000  1792.000000                   29400
            2                      2000  1810.000000                   40500
2026        1                      1280  1808.375000                   17100
            2                      1400  1787.571429                   24800
            3                      1800  1809.000000                   37950

### 9

Select September 2026 renewal records with `offered_discount` greater than zero. Group by `renewal_number` and return:

- `discounted_offers`: number of records.
- `average_discount`: mean offered discount.
- `minimum_discount`: smallest offered discount.
- `maximum_discount`: largest offered discount.

Use `.loc` on the summary to keep only groups with at least 50 discounted offers.

In [73]:
renw = renewals.loc[(renewals['expiry_month'] == 9) & (renewals['expiry_year'] == 2026) & (renewals['offered_discount'] > 0)].groupby('renewal_number').agg(discounted_offers = ('renewal_id', 'count'),
                                                                                                                                 average_discount = ('offered_discount', 'mean'),
                                                                                                                                 minimum_discount = ('offered_discount', 'min'),
                                                                                                                                 maximum_discount = ('offered_discount', 'max'))
rnew = renw.loc[renw['discounted_offers'] >= 50]
rnew

,discounted_offers,average_discount,minimum_discount,maximum_discount
renewal_number,,,,
1,94,181.914894,150,250
2,140,177.142857,150,250
3,211,179.857820,150,250


### 10

Select records whose expiry year is 2026 and whose `renewal_deadline` is before 10 October 2026. Group by `renewal_number` and return:

- `members_due`: number of records.
- `earliest_deadline`: earliest deadline.
- `latest_deadline`: latest deadline.

Explain why this filter excludes renewal windows still open at the end of 9 October. Do not calculate renewal rates: this table alone does not contain successful payment outcomes.

In [74]:
renw = renewals.loc[(renewals['renewal_deadline'] < '2026-10-10'	) & (renewals['expiry_year'] == 2026)].groupby('renewal_number').agg(members_due = ('renewal_id', 'count'),
                                                                                                                                 earliest_deadline = ('renewal_deadline','min'),
                                                                                                                                      latest_deadline = ('renewal_deadline','max'))
#rnew = renw.loc[renw['discounted_offers'] >= 50]
renw

,members_due,earliest_deadline,latest_deadline
renewal_number,,,
1,1280,2026-09-08,2026-10-07
2,1400,2026-09-08,2026-10-07
3,1800,2026-09-08,2026-10-07


### 11

Using `events`, group by `event_type` and return:

- `event_count`: number of events.
- `distinct_members`: number of distinct members.
- `distinct_renewals`: number of distinct renewal IDs.
- `total_event_cost`: sum of recorded event costs.

Why can `event_count` exceed `distinct_members`?

In [75]:
events.head()

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13


In [76]:
events.groupby('event_type').agg(event_count = ('event_id', 'count'),
                                 distinct_members = ('member_id', 'nunique'),
                                 distinct_renewals = ('renewal_id', 'nunique'),
                                 total_event_cost = ('event_cost', 'sum'))

,event_count,distinct_members,distinct_renewals,total_event_cost
event_type,,,,
BENEFIT_USED,125995,6374,11494,4526444
PAYMENT_ATTEMPT,13308,5004,9820,116856
REMINDER_SENT,11580,6380,11580,11580
SUPPORT_CONTACT,1174,1174,1174,52830


### 12

Select only `"PAYMENT_ATTEMPT"` events with outcome `"SUCCESS"`. Group by `channel` and return:

- `successful_payments`: number of successful payment events.
- `cash_collected`: sum of payment amounts.
- `average_payment`: mean payment amount.
- `total_payment_event_cost`: sum of recorded costs for these successful events.

The cost total here excludes failed payment attempts and other event types.

In [77]:
events.loc[(events['event_type'] == 'PAYMENT_ATTEMPT') & (events['outcome'] == 'SUCCESS')].groupby('channel').agg(successful_payments = ('event_id', 'count'),
                                 cash_collected = ('amount', 'sum'),
                                 average_payment = ('amount', 'mean'),
                                 total_payment_event_cost = ('event_cost', 'sum'))

,successful_payments,cash_collected,average_payment,total_payment_event_cost
channel,,,,
APP,4522,8057578,1781.861566,54264
WEB,4502,7989698,1774.699689,54024


### 13

Select failed payment-attempt events. Group by both `channel` and `detail_code`. Return:

- `failed_attempts`: number of failed events.
- `affected_members`: number of distinct members.
- `affected_renewals`: number of distinct renewal IDs.
- `total_failure_cost`: sum of recorded costs for these failed events.

Using `.loc` on the aggregated result, keep only groups with at least 100 failed attempts and at least 40 affected members.

In [79]:
eve = events.loc[(events['event_type'] == 'PAYMENT_ATTEMPT') & (events['outcome'] == 'FAILED')].groupby(['channel','detail_code']).agg(failed_attempts = ('event_id', 'count'),
                                 affected_members = ('member_id', 'nunique'),
                                 affected_renewals = ('renewal_id', 'nunique'),
                                 total_failure_cost = ('event_cost', 'sum'))
eve.loc[(eve['failed_attempts'] >= 100) & (eve['affected_members'] >= 40)]

failed_attempts  affected_members  \
channel detail_code                                                
APP     AUTHENTICATION_FAILED              704               636   
        INSUFFICIENT_FUNDS                 502               482   
        PSP_TIMEOUT                       1717              1048   
WEB     AUTHENTICATION_FAILED              401               391   
        INSUFFICIENT_FUNDS                 367               361   
        PSP_TIMEOUT                        593               558   

                               affected_renewals  total_failure_cost  
channel detail_code                                                   
APP     AUTHENTICATION_FAILED                645                1408  
        INSUFFICIENT_FUNDS                   492                1004  
        PSP_TIMEOUT                         1067                3434  
WEB     AUTHENTICATION_FAILED                400                 802  
        INSUFFICIENT_FUNDS                   366                 734  
        PSP_TIMEOUT                          569                1186

### 14

Select `"BENEFIT_USED"` events recorded through `"APP"` or `"WEB"`. Group by `detail_code` and return:

- `uses`: number of benefit-use events.
- `members_using`: number of distinct members.
- `total_cost`: sum of recorded event costs.
- `average_cost_per_use`: mean event cost.

Using `.loc` on the summary, display only benefits whose average cost per use exceeds 30. Show only `uses`, `members_using` and `average_cost_per_use`.

In [87]:
eve = events.loc[(events['event_type'] == 'BENEFIT_USED') & events['channel'].isin(['WEB','APP'])].groupby('detail_code').agg(uses = ('event_id', 'count'),
                                 members_using = ('member_id', 'nunique'),
                                 total_cost = ('event_cost', 'sum'),
                                 average_cost_per_use = ('event_cost','mean'))
eve.loc[(eve['average_cost_per_use'] > 30),['uses', 'members_using','average_cost_per_use']]


,uses,members_using,average_cost_per_use
detail_code,,,
FREE_DELIVERY,81803,6360,45.057137


### 15

Select `"SUPPORT_CONTACT"` events whose detail code is either `"FREE_DELIVERY_UNAVAILABLE"` or `"RENEWAL_HELP"`. Group by `detail_code` and return:

- `contacts`: number of support contacts.
- `members_contacting`: number of distinct members.
- `renewals_involved`: number of distinct renewal IDs.
- `total_support_cost`: sum of recorded support costs.

Explain why the number of members contacting support cannot automatically be treated as the number of lost members.

In [88]:
eve = events.loc[(events['event_type'] == 'SUPPORT_CONTACT') & events['detail_code'].isin(['FREE_DELIVERY_UNAVAILABLE','RENEWAL_HELP'])].groupby('detail_code').agg(contacts = ('event_id', 'count'),
                                 members_contacting = ('member_id', 'nunique'),
                                 renewals_involved = ('renewal_id', 'nunique'),
                                 total_support_cost = ('event_cost', 'sum'))
eve

,contacts,members_contacting,renewals_involved,total_support_cost
detail_code,,,,
FREE_DELIVERY_UNAVAILABLE,620,620,620,27900
RENEWAL_HELP,104,104,104,4680
